# Priority Queues and Heaps

```{contents}
:local:
:depth: 2
```

## Priority Queue Behavior

```{index} priority queue
```


A **priority queue** removes the item with the next priority, not necessarily the item that arrived first. In C#'s `PriorityQueue<TElement, TPriority>`, lower priority values come out first by default.


In [ ]:
using System;
using System.Collections.Generic;

var tasks = new PriorityQueue<string, int>();
tasks.Enqueue("Process invoice", 2);
tasks.Enqueue("Respond to outage", 0);
tasks.Enqueue("Send summary", 3);
tasks.Enqueue("Reset password", 1);

while (tasks.Count > 0)
{
    string task = tasks.Dequeue();
    Console.WriteLine(task);
}


The outage has priority `0`, so it is processed before lower-urgency work. Priority queues are useful for schedulers, event simulations, pathfinding, triage, and any workflow where "next" means most important rather than oldest.


A priority queue defines how you select the next element. It does not promise that enumeration is sorted or that arbitrary keys are easy to locate. The numerical priority has an application meaning: here zero is most urgent. Dequeue from an empty queue is an error; `TryDequeue` reports false instead.

Equal priorities need a policy. The .NET library does not guarantee arrival order for ties. If the service desk requires first-arrived service among equally urgent tickets, use the pair `(urgency, arrivalNumber)` as the priority. Tuple comparison considers urgency first and arrival second. Each arrival number is unique within the queue, so the resulting order is deterministic.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var queue = new PriorityQueue<string, (int Urgency, long Arrival)>();
long arrival = 0;
queue.Enqueue("T-100", (2, arrival++));
queue.Enqueue("T-101", (0, arrival++));
queue.Enqueue("T-102", (2, arrival++));
while (queue.TryDequeue(out string? ticket, out var priority))
    Console.WriteLine($"{ticket}: urgency={priority.Urgency}, arrival={priority.Arrival}");
Console.WriteLine($"empty={queue.Count == 0}");

The order is T-101, T-100, T-102. A later urgent ticket can move ahead, while equal-urgency tickets retain their arrival order. The queue alone does not prevent starvation if urgent work arrives indefinitely. Fairness requires an additional scheduling policy.

The manual implementation in this section is a **binary** heap. The current .NET `PriorityQueue` uses four children per parent internally. Its API behavior is the same priority-queue idea, but the binary index formulas below describe the teaching implementation, not the library's internal layout. [Microsoft's PriorityQueue documentation](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2) describes both tie behavior and the library representation.

## Heap Representation

```{index} heap; min-heap; complete tree
```


A **heap** is a common priority-queue representation. A binary heap has two key rules:

- **Shape rule**: the tree is complete, filled level by level from left to right.
- **Order rule**: in a min-heap, each parent priority is less than or equal to its child priorities.

Because the shape is complete, a heap can be stored compactly in an array. For an item at index `i`, the children are at `2 * i + 1` and `2 * i + 2`.


In [ ]:
using System;

int[] heap = { 1, 3, 2, 7, 5, 6, 4 };

for (int parent = 0; parent < heap.Length; parent++)
{
    int left = 2 * parent + 1;
    int right = 2 * parent + 2;

    if (left < heap.Length)
    {
        Console.WriteLine($"{heap[parent]} -> left child {heap[left]}");
    }

    if (right < heap.Length)
    {
        Console.WriteLine($"{heap[parent]} -> right child {heap[right]}");
    }
}


The array is not fully sorted. It only guarantees that each parent is no larger than its children. That weaker rule is enough to keep the minimum at the root.


For a non-root index i, its parent is `(i - 1) / 2`, using integer division. Valid child indexes must be less than Count. Count describes occupied positions; the backing array may have extra capacity that is not part of the heap.

```text
          1 (0)
        /       \
     3 (1)     2 (2)
    /    \     /    \
 7 (3) 5 (4) 6 (5) 4 (6)
```

The values obey parent-child order, but siblings and separate subtrees can be out of numerical order. For example, 3 precedes 2 in the array. Repeatedly removing the root produces a sorted sequence; reading the raw array does not. The complete shape guarantees height Θ(log n) for a nonempty binary heap.

## Enqueue and Bubble Up

```{index} heap; bubble up
```


To insert into a heap, place the new item at the end to preserve the complete-tree shape, then **bubble up** while it violates the heap order rule.


In [ ]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
heap.Add(0);

int child = heap.Count - 1;
while (child > 0)
{
    int parent = (child - 1) / 2;
    if (heap[parent] <= heap[child])
    {
        break;
    }

    (heap[parent], heap[child]) = (heap[child], heap[parent]);
    child = parent;
}

Console.WriteLine(string.Join(", ", heap));


The new value `0` starts at the end. It swaps with larger parents until the min-heap property is restored.


Trace the appended zero from index seven to the root:

| Stage | Index path | Array |
| --- | --- | --- |
| Append | 7 | 1, 3, 2, 7, 5, 6, 4, 0 |
| Swap with 7 | 7 → 3 | 1, 3, 2, 0, 5, 6, 4, 7 |
| Swap with 3 | 3 → 1 | 1, 0, 2, 3, 5, 6, 4, 7 |
| Swap with 1 | 1 → 0 | 0, 1, 2, 3, 5, 6, 4, 7 |

Before insertion, every old parent-child relation satisfies the order rule. Appending adds only one possible violation between the new item and its parent. Each swap repairs that relation and moves the possible violation upward. Stop at the root or when the parent is no larger. This explains why repairing one path is sufficient.

## Dequeue and Sink Down

```{index} heap; sink down
```


To remove the minimum item, remove the root, move the last item to the root, then **sink down** until the heap order rule is restored.


In [ ]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
int removed = heap[0];
heap[0] = heap[^1];
heap.RemoveAt(heap.Count - 1);

int parent = 0;
while (true)
{
    int left = 2 * parent + 1;
    int right = 2 * parent + 2;
    int smallest = parent;

    if (left < heap.Count && heap[left] < heap[smallest]) smallest = left;
    if (right < heap.Count && heap[right] < heap[smallest]) smallest = right;
    if (smallest == parent) break;

    (heap[parent], heap[smallest]) = (heap[smallest], heap[parent]);
    parent = smallest;
}

Console.WriteLine($"Removed: {removed}");
Console.WriteLine(string.Join(", ", heap));


Both bubble-up and sink-down follow one root-to-leaf path, so their work grows with heap height, not with every item in the heap.


In the removal example, replacing root 1 with the last value 4 gives `4, 3, 2, 7, 5, 6`. Compare both children before choosing the swap: 2 is smaller than 3, so swap indexes zero and two. The result is `2, 3, 4, 7, 5, 6`. The next child is 6; 4 is already no larger, so stop.

Choosing a larger child can leave the root larger than the other child. Sink-down must choose the smaller existing child and handle a last parent with only a left child. Removing the last array position preserves the complete shape. For a singleton, removal leaves no root to repair.

The complete generic heap below hides the backing list and accepts an optional comparer. The element type must support the default ordering, or you must supply a comparer. `Peek` and `Dequeue` throw on empty; `TryDequeue` distinguishes empty from a stored default value. Equal values are legal, but this heap does not impose stable arrival order on equal comparisons.

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

var priorities = new BinaryMinHeap<int>();
foreach (int urgency in new[] { 3, 0, 2, 0, 5 }) priorities.Enqueue(urgency);
Console.WriteLine($"count={priorities.Count}, minimum={priorities.Peek()}, valid={priorities.IsValid()}");
Console.WriteLine("raw=" + string.Join(", ", priorities.Snapshot()));
while (priorities.TryDequeue(out int urgency)) Console.WriteLine(urgency);
priorities.Enqueue(0);
Console.WriteLine($"reused zero={priorities.Dequeue()}, empty={priorities.Count == 0}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

`List<T>` grows its storage when necessary. Bubble-up uses O(log n) comparisons, but a single enqueue that copies a full backing array can take O(n) time. Across a long sequence, storage growth contributes amortized O(1) per append; enqueue therefore has amortized O(log(n + 1)) time. Dequeue repairs at most one downward path and takes O(log(n + 1)); it does not shrink this backing list. Peek and Count take O(1). The iterative repair uses O(1) extra space, while the stored items use O(n) space.

`IsValid` scans all parent-child relations, so it takes Θ(n) time. It is a diagnostic method, not part of every production operation. `Snapshot` also takes Θ(n) time and space to copy the occupied values. Checking the minimum alone would miss a violation inside a subtree. Test empty and singleton states, duplicate priorities, an odd final child, ascending/descending input, draining, and reuse.

A heap can also be built from an existing array by sinking internal nodes from right to left. Most nodes are near leaves and require little repair; summing that work gives O(n) bottom-up construction. Repeated insertion has an O(n log n) worst-case bound. This chapter implements repeated enqueue first; bottom-up construction is an extension.

## Exercise

```{index} priority queue; exercise
```


In [ ]:
// Exercise: Triage tickets
// Enqueue three tickets with priorities. Lower number means more urgent.
// Print them in priority order.

using System;
using System.Collections.Generic;

var tickets = new PriorityQueue<string, int>();

// Your code starts here.



// Your code ends here.


In [1]:
// Solution
using System;
using System.Collections.Generic;

var tickets = new PriorityQueue<string, int>();
tickets.Enqueue("Password reset", 3);
tickets.Enqueue("Payment outage", 0);
tickets.Enqueue("New laptop request", 5);

while (tickets.Count > 0)
{
    Console.WriteLine(tickets.Dequeue());
}


Payment outage
Password reset
New laptop request


## Summary

```{index} Summary
```


A priority queue is the behavior; a heap is one efficient representation. C#'s `PriorityQueue<TElement, TPriority>` is a min-priority queue by default, which makes lower priority values come out first.

```{rubric} Footnotes
```

[^1]: `PriorityQueue<TElement, TPriority>` was added to .NET in modern versions of the base class library; older C# material often implements heaps manually.
